<a href="https://colab.research.google.com/github/reterman/semi-automatic-music-transcription-system/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎶 **YourMT3+**  
"YourMT3+: Multi-instrument Music Transcription with Enhanced Transformer Architectures and Cross-dataset Stem Augmentation" <br>
<i>Sungkyun Chang, Emmanouil Benetos, Holger Kirchhoff and Simon Dixon,  <a href="https://arxiv.org/abs/2407.04822">IEEE MLSP 2024</a> </i>

<div>
<img src="https://i.imgur.com/yfa53Xn.jpeg" width="800" />
</div>



#### 🥁 How to use:
1. Execute the code blocks below in sequence.
2. In the GradIO interface, select either 'Audio upload' or 'YouTube' via the tabs.
3. Click on "Play" or "Transcribe".

#### Known Issues:
- After changing the model checkpoint, a restart is required.
- When transcribing music from sources outside the dataset, models trained with Pitch-shift (PS) often incorrectly transcribe segments a semitone higher or lower. This issue is not observed in the YPTF-S (noPS) model, as seen in the example at https://youtu.be/9E82wwNc7r8?si=I-WyfwJXCBDY2reh.

## SETUP MODEL

In [1]:
!pip list | grep torch

pytorch-lightning                        2.6.1
torch                                    2.8.0
torchao                                  0.10.0
torchaudio                               2.8.0
torchcodec                               0.10.0+cu128
torchdata                                0.11.0
torchmetrics                             1.9.0
torchsummary                             1.5.1
torchtune                                0.6.1
torchvision                              0.23.0


torch==2.8

torchaudio==2.8.0

torchvision==0.23.0

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install torchaudio==2.8.0
!pip install torchvision==0.23.0

In [ ]:
# @title Setup
!pip install awscli
!
!aws s3 sync s3://amt-deploy-public/amt/ /content/amt --no-sign-request
!aws s3 sync s3://amt-deploy-public/examples/ /content/examples --no-sign-request
%cd amt/src
!pip install transformers==4.45.1
!apt-get install sox
!pip install gradio
!pip install -r requirements.txt
!python -m pip install -U yt-dlp[default]

/content/amt/src
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
sox is already the newest version (14.4.2+git20190427-2+deb11u2ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 100 not upgraded.
  Cloning https://github.com/craffel/mir_eval.git to /tmp/pip-req-build-5j6wq97b
  Running command git clone --filter=blob:none --quiet https://github.com/craffel/mir_eval.git /tmp/pip-req-build-5j6wq97b
  Resolved https://github.com/craffel/mir_eval.git to commit fe73b3533737814f83dbd9739f06e90f5f82f758
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/katsura-jp/pytorch-cosine-annealing-with-warmup.git to /tmp/pip-req-build-4fpr9u2k
  Running command git clone --filter=blob:none --quiet https://github.com/katsura-jp/pytorch-cosine-annealing-with-warmup.git /tmp/pip-req-build-4fpr9u2k
  Resolved https://github.com/katsura-jp/pytorch-cosine-annealing-with-warmup.git to commit 12d03c07553aedd3d9e9155e2b3e31ce8c6

In [ ]:
# @title Model helper
%cd /content/amt/src
from collections import Counter
import argparse
import torch
import numpy as np
import pretty_midi

from model.init_train import initialize_trainer, update_config
from utils.task_manager import TaskManager
from config.vocabulary import drum_vocab_presets
from utils.utils import str2bool
from utils.utils import Timer
from utils.audio import slice_padded_array
from utils.note2event import mix_notes
from utils.event2note import merge_zipped_note_events_and_ties_to_notes
from utils.utils import write_model_output_as_midi, write_err_cnt_as_json
from model.ymt3 import YourMT3

def load_model_checkpoint(args=None):
    parser = argparse.ArgumentParser(description="YourMT3")
    # General
    parser.add_argument('exp_id', type=str, help='A unique identifier for the experiment is used to resume training. The "@" symbol can be used to load a specific checkpoint.')
    parser.add_argument('-p', '--project', type=str, default='ymt3', help='project name')
    parser.add_argument('-ac', '--audio-codec', type=str, default=None, help='audio codec (default=None). {"spec", "melspec"}. If None, default value defined in config.py will be used.')
    parser.add_argument('-hop', '--hop-length', type=int, default=None, help='hop length in frames (default=None). {128, 300} 128 for MT3, 300 for PerceiverTFIf None, default value defined in config.py will be used.')
    parser.add_argument('-nmel', '--n-mels', type=int, default=None, help='number of mel bins (default=None). If None, default value defined in config.py will be used.')
    parser.add_argument('-if', '--input-frames', type=int, default=None, help='number of audio frames for input segment (default=None). If None, default value defined in config.py will be used.')
    # Model configurations
    parser.add_argument('-sqr', '--sca-use-query-residual', type=str2bool, default=None, help='sca use query residual flag. Default follows config.py')
    parser.add_argument('-enc', '--encoder-type', type=str, default=None, help="Encoder type. 't5' or 'perceiver-tf' or 'conformer'. Default is 't5', following config.py.")
    parser.add_argument('-dec', '--decoder-type', type=str, default=None, help="Decoder type. 't5' or 'multi-t5'. Default is 't5', following config.py.")
    parser.add_argument('-preenc', '--pre-encoder-type', type=str, default='default', help="Pre-encoder type. None or 'conv' or 'default'. By default, t5_enc:None, perceiver_tf_enc:conv, conformer:None")
    parser.add_argument('-predec', '--pre-decoder-type', type=str, default='default', help="Pre-decoder type. {None, 'linear', 'conv1', 'mlp', 'group_linear'} or 'default'. Default is {'t5': None, 'perceiver-tf': 'linear', 'conformer': None}.")
    parser.add_argument('-cout', '--conv-out-channels', type=int, default=None, help='Number of filters for pre-encoder conv layer. Default follows "model_cfg" of config.py.')
    parser.add_argument('-tenc', '--task-cond-encoder', type=str2bool, default=True, help='task conditional encoder (default=True). True or False')
    parser.add_argument('-tdec', '--task-cond-decoder', type=str2bool, default=True, help='task conditional decoder (default=True). True or False')
    parser.add_argument('-df', '--d-feat', type=int, default=None, help='Audio feature will be projected to this dimension for Q,K,V of T5 or K,V of Perceiver (default=None). If None, default value defined in config.py will be used.')
    parser.add_argument('-pt', '--pretrained', type=str2bool, default=False, help='pretrained T5(default=False). True or False')
    parser.add_argument('-b', '--base-name', type=str, default="google/t5-v1_1-small", help='base model name (default="google/t5-v1_1-small")')
    parser.add_argument('-epe', '--encoder-position-encoding-type', type=str, default='default', help="Positional encoding type of encoder. By default, pre-defined PE for T5 or Perceiver-TF encoder in config.py. For T5: {'sinusoidal', 'trainable'}, conformer: {'rotary', 'trainable'}, Perceiver-TF: {'trainable', 'rope', 'alibi', 'alibit', 'None', '0', 'none', 'tkd', 'td', 'tk', 'kdt'}.")
    parser.add_argument('-dpe', '--decoder-position-encoding-type', type=str, default='default', help="Positional encoding type of decoder. By default, pre-defined PE for T5 in config.py. {'sinusoidal', 'trainable'}.")
    parser.add_argument('-twe', '--tie-word-embedding', type=str2bool, default=None, help='tie word embedding (default=None). If None, default value defined in config.py will be used.')
    parser.add_argument('-el', '--event-length', type=int, default=None, help='event length (default=None). If None, default value defined in model cfg of config.py will be used.')
    # Perceiver-TF configurations
    parser.add_argument('-dl', '--d-latent', type=int, default=None, help='Latent dimension of Perceiver. On T5, this will be ignored (default=None). If None, default value defined in config.py will be used.')
    parser.add_argument('-nl', '--num-latents', type=int, default=None, help='Number of latents of Perceiver. On T5, this will be ignored (default=None). If None, default value defined in config.py will be used.')
    parser.add_argument('-dpm', '--perceiver-tf-d-model', type=int, default=None, help='Perceiver-TF d_model (default=None). If None, default value defined in config.py will be used.')
    parser.add_argument('-npb', '--num-perceiver-tf-blocks', type=int, default=None, help='Number of blocks of Perceiver-TF. On T5, this will be ignored (default=None). If None, default value defined in config.py.')
    parser.add_argument('-npl', '--num-perceiver-tf-local-transformers-per-block', type=int, default=None, help='Number of local layers per block of Perceiver-TF. On T5, this will be ignored (default=None). If None, default value defined in config.py will be used.')
    parser.add_argument('-npt', '--num-perceiver-tf-temporal-transformers-per-block', type=int, default=None, help='Number of temporal layers per block of Perceiver-TF. On T5, this will be ignored (default=None). If None, default value defined in config.py will be used.')
    parser.add_argument('-atc', '--attention-to-channel', type=str2bool, default=None, help='Attention to channel flag of Perceiver-TF. On T5, this will be ignored (default=None). If None, default value defined in config.py will be used.')
    parser.add_argument('-ln', '--layer-norm-type', type=str, default=None, help='Layer normalization type (default=None). {"layer_norm", "rms_norm"}. If None, default value defined in config.py will be used.')
    parser.add_argument('-ff', '--ff-layer-type', type=str, default=None, help='Feed forward layer type (default=None). {"mlp", "moe", "gmlp"}. If None, default value defined in config.py will be used.')
    parser.add_argument('-wf', '--ff-widening-factor', type=int, default=None, help='Feed forward layer widening factor for MLP/MoE/gMLP (default=None). If None, default value defined in config.py will be used.')
    parser.add_argument('-nmoe', '--moe-num-experts', type=int, default=None, help='Number of experts for MoE (default=None). If None, default value defined in config.py will be used.')
    parser.add_argument('-kmoe', '--moe-topk', type=int, default=None, help='Top-k for MoE (default=None). If None, default value defined in config.py will be used.')
    parser.add_argument('-act', '--hidden-act', type=str, default=None, help='Hidden activation function (default=None). {"gelu", "silu", "relu", "tanh"}. If None, default value defined in config.py will be used.')
    parser.add_argument('-rt', '--rotary-type', type=str, default=None, help='Rotary embedding type expressed in three letters. e.g. ppl: "pixel" for SCA and latents, "lang" for temporal transformer. If None, use config.')
    parser.add_argument('-rk', '--rope-apply-to-keys', type=str2bool, default=None, help='Apply rope to keys (default=None). If None, use config.')
    parser.add_argument('-rp', '--rope-partial-pe', type=str2bool, default=None, help='Whether to apply RoPE to partial positions (default=None). If None, use config.')
    # Decoder configurations
    parser.add_argument('-dff', '--decoder-ff-layer-type', type=str, default=None, help='Feed forward layer type of decoder (default=None). {"mlp", "moe", "gmlp"}. If None, default value defined in config.py will be used.')
    parser.add_argument('-dwf', '--decoder-ff-widening-factor', type=int, default=None, help='Feed forward layer widening factor for decoder MLP/MoE/gMLP (default=None). If None, default value defined in config.py will be used.')
    # Task and Evaluation configurations
    parser.add_argument('-tk', '--task', type=str, default='mt3_full_plus', help='tokenizer type (default=mt3_full_plus). See config/task.py for more options.')
    parser.add_argument('-epv', '--eval-program-vocab', type=str, default=None, help='evaluation vocabulary (default=None). If None, default vocabulary of the data preset will be used.')
    parser.add_argument('-edv', '--eval-drum-vocab', type=str, default=None, help='evaluation vocabulary for drum (default=None). If None, default vocabulary of the data preset will be used.')
    parser.add_argument('-etk', '--eval-subtask-key', type=str, default='default', help='evaluation subtask key (default=default). See config/task.py for more options.')
    parser.add_argument('-t', '--onset-tolerance', type=float, default=0.05, help='onset tolerance (default=0.05).')
    parser.add_argument('-os', '--test-octave-shift', type=str2bool, default=False, help='test optimal octave shift (default=False). True or False')
    parser.add_argument('-w', '--write-model-output', type=str2bool, default=True, help='write model test output to file (default=False). True or False')
    # Trainer configurations
    parser.add_argument('-pr','--precision', type=str, default="bf16-mixed", help='precision (default="bf16-mixed") {32, 16, bf16, bf16-mixed}')
    parser.add_argument('-st', '--strategy', type=str, default='auto', help='strategy (default=auto). auto or deepspeed or ddp')
    parser.add_argument('-n', '--num-nodes', type=int, default=1, help='number of nodes (default=1)')
    parser.add_argument('-g', '--num-gpus', type=str, default='auto', help='number of gpus (default="auto")')
    parser.add_argument('-wb', '--wandb-mode', type=str, default="disabled", help='wandb mode for logging (default=None). "disabled" or "online" or "offline". If None, default value defined in config.py will be used.')
    # Debug
    parser.add_argument('-debug', '--debug-mode', type=str2bool, default=False, help='debug mode (default=False). True or False')
    parser.add_argument('-tps', '--test-pitch-shift', type=int, default=None, help='use pitch shift when testing. debug-purpose only. (default=None). semitone in int.')
    args = parser.parse_args(args)
    # yapf: enable
    if torch.__version__ >= "1.13":
        torch.set_float32_matmul_precision("high")
    args.epochs = None

    # Initialize and update config
    _, _, dir_info, shared_cfg = initialize_trainer(args, stage='test')
    shared_cfg, audio_cfg, model_cfg = update_config(args, shared_cfg, stage='test')

    if args.eval_drum_vocab != None:  # override eval_drum_vocab
        eval_drum_vocab = drum_vocab_presets[args.eval_drum_vocab]

    # Initialize task manager
    tm = TaskManager(task_name=args.task,
                     max_shift_steps=int(shared_cfg["TOKENIZER"]["max_shift_steps"]),
                     debug_mode=args.debug_mode)
    print(f"Task: {tm.task_name}, Max Shift Steps: {tm.max_shift_steps}")

    # Use GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Model
    model = YourMT3(
        audio_cfg=audio_cfg,
        model_cfg=model_cfg,
        shared_cfg=shared_cfg,
        optimizer=None,
        task_manager=tm,  # tokenizer is a member of task_manager
        eval_subtask_key=args.eval_subtask_key,
        write_output_dir=dir_info["lightning_dir"] if args.write_model_output or args.test_octave_shift else None
        ).to(device)
    checkpoint = torch.load(dir_info["last_ckpt_path"], weights_only=False) # fix model loading error in torch 2.6
    state_dict = checkpoint['state_dict']
    new_state_dict = {k: v for k, v in state_dict.items() if 'pitchshift' not in k}
    model.load_state_dict(new_state_dict, strict=False)
    return model.eval()


def transcribe(model, audio_info):
    t = Timer()

    # Converting Audio
    t.start()
    audio, sr = torchaudio.load(uri=audio_info['filepath'])
    audio = torch.mean(audio, dim=0).unsqueeze(0)
    audio = torchaudio.functional.resample(audio, sr, model.audio_cfg['sample_rate'])
    audio_segments = slice_padded_array(audio, model.audio_cfg['input_frames'], model.audio_cfg['input_frames'])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    audio_segments = torch.from_numpy(audio_segments.astype('float32')).to(device).unsqueeze(1) # (n_seg, 1, seg_sz)
    t.stop(); t.print_elapsed_time("converting audio");

    # Inference
    t.start()
    pred_token_arr, _ = model.inference_file(bsz=8, audio_segments=audio_segments)
    t.stop(); t.print_elapsed_time("model inference");

    # Post-processing
    t.start()
    num_channels = model.task_manager.num_decoding_channels
    n_items = audio_segments.shape[0]
    start_secs_file = [model.audio_cfg['input_frames'] * i / model.audio_cfg['sample_rate'] for i in range(n_items)]
    pred_notes_in_file = []
    n_err_cnt = Counter()
    for ch in range(num_channels):
        pred_token_arr_ch = [arr[:, ch, :] for arr in pred_token_arr]  # (B, L)
        zipped_note_events_and_tie, list_events, ne_err_cnt = model.task_manager.detokenize_list_batches(
            pred_token_arr_ch, start_secs_file, return_events=True)
        pred_notes_ch, n_err_cnt_ch = merge_zipped_note_events_and_ties_to_notes(zipped_note_events_and_tie)
        pred_notes_in_file.append(pred_notes_ch)
        n_err_cnt += n_err_cnt_ch
    pred_notes = mix_notes(pred_notes_in_file)  # This is the mixed notes from all channels

    # Write MIDI
    write_model_output_as_midi(pred_notes, '/content/',
                              audio_info['track_name'], model.midi_output_inverse_vocab)
    t.stop(); t.print_elapsed_time("post processing");
    midifile =  os.path.join('/content/model_output/', audio_info['track_name']  + '.mid')
    assert os.path.exists(midifile)
    return midifile


In [ ]:
# @title GradIO helper
import os
import subprocess
import glob
from typing import Tuple, Dict, Literal
from ctypes import ArgumentError
from google.colab import output

import gradio as gr
import torchaudio

def prepare_media(source_path_or_url: os.PathLike,
                  source_type: Literal['audio_filepath', 'youtube_url'],
                  delete_video: bool = True) -> Dict:
    """prepare media from source path or youtube, and return audio info"""
    # Get audio_file
    if source_type == 'audio_filepath':
        audio_file = source_path_or_url
    elif source_type == 'youtube_url':
        # Download from youtube
        try:
            # Try PyTube first
            proxy_handler = {"http": "http://127.0.0.1:1087", "https":"http://127.0.0.1:1087"}
            yt = YouTube(source_path_or_url, proxies=proxy_handler)
            # yt = YouTube(source_path_or_url)
            audio_stream = min(yt.streams.filter(only_audio=True), key=lambda s: s.bitrate)
            mp4_file = audio_stream.download(output_path='downloaded') # ./downloaded
            audio_file = mp4_file[:-3] + 'mp3'
            subprocess.run(['ffmpeg', '-i', mp4_file, '-ac', '1', audio_file])
            os.remove(mp4_file)
        except Exception as e:
            try:
                # Try alternative
                print(f"Failed with PyTube, error: {e}. Trying yt-dlp...")
                audio_file = './downloaded/yt_audio'
                subprocess.run(['yt-dlp', '-x', source_path_or_url, '-f', 'bestaudio',
                    '-o', audio_file, '--audio-format', 'mp3', '--restrict-filenames',
                    '--force-overwrites'])
                # subprocess.run(['yt-dlp', '-x', source_path_or_url, '-f', 'bestaudio',
                #     '-o', audio_file, '--audio-format', 'mp3', '--restrict-filenames',
                #     '--force-overwrites', '--cookiefile', '/content/cookies.txt'])
                audio_file += '.mp3'
            except Exception as e:
                print(f"Alternative downloader failed, error: {e}. Please try again later!")
                return None
    else:
        raise ValueError(source_type)

    # Create info
    info = torchaudio.info(audio_file)
    return {
        "filepath": audio_file,
        "track_name": os.path.basename(audio_file).split('.')[0],
        "sample_rate": int(info.sample_rate),
        "bits_per_sample": int(info.bits_per_sample),
        "num_channels": int(info.num_channels),
        "num_frames": int(info.num_frames),
        "duration": int(info.num_frames / info.sample_rate),
        "encoding": str.lower(info.encoding),
        }

def process_audio(audio_filepath):
    if audio_filepath is None:
        return None
    audio_info = prepare_media(audio_filepath, source_type='audio_filepath')
    midifile = transcribe(model, audio_info)
    midifile = to_data_url(midifile)
    return create_html_from_midi(midifile) # html midiplayer

def process_video(youtube_url):
    if 'youtu' not in youtube_url:
        return None
    audio_info = prepare_media(youtube_url, source_type='youtube_url')
    midifile = transcribe(model, audio_info)
    midifile = to_data_url(midifile)
    return create_html_from_midi(midifile) # html midiplayer

def play_video(youtube_url):
    if 'youtu' not in youtube_url:
        return None
    return create_html_youtube_player(youtube_url)


In [ ]:
# @title Load Checkpoint
model_name = 'YMT3+' # @param ["YMT3+", "YPTF+Single (noPS)", "YPTF+Multi (PS)", "YPTF.MoE+Multi (noPS)", "YPTF.MoE+Multi (PS)"]
precision = '32' # @param ["32", "bf16-mixed", "16"]
project = '2024'

if model_name == "YMT3+":
    checkpoint = "notask_all_cross_v6_xk2_amp0811_gm_ext_plus_nops_b72@model.ckpt"
    args = [checkpoint, '-p', project, '-pr', precision]
elif model_name == "YPTF+Single (noPS)":
    checkpoint = "ptf_all_cross_rebal5_mirst_xk2_edr005_attend_c_full_plus_b100@model.ckpt"
    args = [checkpoint, '-p', project, '-enc', 'perceiver-tf', '-ac', 'spec',
            '-hop', '300', '-atc', '1', '-pr', precision]
elif model_name == "YPTF+Multi (PS)":
    checkpoint = "mc13_256_all_cross_v6_xk5_amp0811_edr005_attend_c_full_plus_2psn_nl26_sb_b26r_800k@model.ckpt"
    args = [checkpoint, '-p', project, '-tk', 'mc13_full_plus_256',
            '-dec', 'multi-t5', '-nl', '26', '-enc', 'perceiver-tf',
            '-ac', 'spec', '-hop', '300', '-atc', '1', '-pr', precision]
elif model_name == "YPTF.MoE+Multi (noPS)":
    checkpoint = "mc13_256_g4_all_v7_mt3f_sqr_rms_moe_wf4_n8k2_silu_rope_rp_b36_nops@last.ckpt"
    args = [checkpoint, '-p', project, '-tk', 'mc13_full_plus_256', '-dec', 'multi-t5',
            '-nl', '26', '-enc', 'perceiver-tf', '-sqr', '1', '-ff', 'moe',
            '-wf', '4', '-nmoe', '8', '-kmoe', '2', '-act', 'silu', '-epe', 'rope',
            '-rp', '1', '-ac', 'spec', '-hop', '300', '-atc', '1', '-pr', precision]
elif model_name == "YPTF.MoE+Multi (PS)":
    checkpoint = "mc13_256_g4_all_v7_mt3f_sqr_rms_moe_wf4_n8k2_silu_rope_rp_b80_ps2@model.ckpt"
    args = [checkpoint, '-p', project, '-tk', 'mc13_full_plus_256', '-dec', 'multi-t5',
            '-nl', '26', '-enc', 'perceiver-tf', '-sqr', '1', '-ff', 'moe',
            '-wf', '4', '-nmoe', '8', '-kmoe', '2', '-act', 'silu', '-epe', 'rope',
            '-rp', '1', '-ac', 'spec', '-hop', '300', '-atc', '1', '-pr', precision]
else:
    raise ValueError(model_name)

print(args)
model = load_model_checkpoint(args=args)




## MAGIC

In [ ]:
audio_filepath = "/content/drive/MyDrive/AMT/8/8.wav"
audio_info = prepare_media(audio_filepath, source_type='audio_filepath')
midifile = transcribe(model, audio_info)

In [ ]:
print(midifile)

In [ ]:
def merge_instruments(midi, program=0):
    # create a single new instrument
    merged_instrument = pretty_midi.Instrument(program=program)

    for inst in midi.instruments:
        for note in inst.notes:
            merged_instrument.notes.append(note)

    # create a new MIDI
    new_midi = pretty_midi.PrettyMIDI()
    new_midi.instruments.append(merged_instrument)

    return(new_midi)


est_midi = pretty_midi.PrettyMIDI(midifile)

for instrument in est_midi.instruments:
    print("Instrument program:", instrument.program)
    # for note in instrument.notes:
        # print(f"Note: pitch={note.pitch}, start={note.start:.2f}, end={note.end:.2f}, velocity={note.velocity}")


combined = merge_instruments(est_midi)

combined.write("/content/drive/MyDrive/AMT/8/8_trans.mid")


In [ ]:
for i in range(11):
  index =i+1
  audio_filepath = "/content/drive/MyDrive/AMT/"+str(index)+"/"+str(index)+".wav"
  audio_info = prepare_media(audio_filepath, source_type='audio_filepath')

  midifile = transcribe(model, audio_info)

  est_midi = pretty_midi.PrettyMIDI(midifile)
  combined = merge_instruments(est_midi)

  midi_filepath = "/content/drive/MyDrive/AMT/"+str(index)+"/"+str(index)+"_est.mid"
  combined.write(midi_filepath)

  print(midi_filepath)


In [ ]:
pip install pretty_midi

In [ ]:
import pretty_midi

pretty_midi.pretty_midi.MAX_TICK = 1e10
midi1 = pretty_midi.PrettyMIDI("/content/drive/MyDrive/AMT/1/1.mid")

midi2 = pretty_midi.PrettyMIDI('/content/drive/MyDrive/AMT/1/1_est.mid')

# create a new MIDI
combined = pretty_midi.PrettyMIDI()

# add instruments from the first file
for inst in midi1.instruments:
    combined.instruments.append(inst)

# add instruments from the second file
for inst in midi2.instruments:
    combined.instruments.append(inst)

combined.write('/content/combined.mid')

## ANALYSIS

In [ ]:
pip install pretty_midi

In [ ]:
import librosa

pretty_midi.pretty_midi.MAX_TICK = 1e10
est_midi = pretty_midi.PrettyMIDI('/content/drive/MyDrive/AMT/8/8.mid')


In [ ]:
def pitch_to_note(pitch: int | float) -> str:
    """MIDI pitch ? 'C4', 'C#4' (using either ? or b)"""
    return librosa.midi_to_note(int(round(pitch)),
                                octave=True,
                                cents=False,
                                key='C:maj',
                                unicode=True)


In [ ]:

def smart_filter_short_notes(midi_data, min_possible_sec=0.02, fallback_percentile=15, append_pitch=1):
    """
    Smart removal of very short notes based on duration distribution analysis.

    :param midi_data: PrettyMIDI object
    :param min_possible_sec: Absolute minimum. Notes shorter than this (e.g. 20 ms) are always removed.
    :param fallback_percentile: If no clear gap is found, trim this % of the shortest notes.
    """
    all_durations = []

    # 1. Collect statistics across all instruments
    for instrument in midi_data.instruments:
        for note in instrument.notes:
            dur = note.end - note.start
            if dur > 0:
                all_durations.append(dur)

    if not all_durations:
        return midi_data

    # Convert to a numpy array for efficient processing
    durations = np.array(all_durations)

    # 2. Compute the threshold

    # Sort durations from shortest to longest
    sorted_dur = np.sort(durations)

    # We only search for a gap among short notes (for example, in the first quarter of the list)
    # This protects us from detecting random gaps between long notes
    search_region_idx = int(len(sorted_dur) * 0.15)
    short_durations = sorted_dur[:search_region_idx]

    if len(short_durations) < 2:
        threshold = np.percentile(durations, fallback_percentile)
    else:
        diffs = np.diff(short_durations)

        # Find the index of the largest jump
        max_gap_idx = np.argmax(diffs)
        max_gap_val = diffs[max_gap_idx]

        # Check whether this jump is significant.
        # For example, if the gap is > 30 ms, we treat it as the boundary between noise and music.
        if max_gap_val:
            # Place the threshold in the middle of the gap
            threshold = short_durations[max_gap_idx]
            # threshold = short_durations[max_gap_idx + 1]

            print(f"[Smart Filter] Found a gap in the distribution. Threshold: {threshold*1000:.1f} ms")
        else:
            # No clear gap was found (most likely the notes are all valid, or the signal is mostly noise)
            threshold = np.percentile(durations, fallback_percentile)
            print(f"[Smart Filter] No gap found. Using the {fallback_percentile}% percentile: {threshold*1000:.1f} ms")

    # 3. Filtering
    # Hard lower bound (acoustics): notes shorter than 20-30 ms are not physically plausible
    final_threshold = max(threshold, min_possible_sec)

    total_notes = 0
    removed_notes = 0

    for instrument in midi_data.instruments:
        original_count = len(instrument.notes)
        new_notes = [n for n in instrument.notes if (n.end - n.start) >= final_threshold]
        for n in new_notes:
            n.pitch = n.pitch + append_pitch

        instrument.notes = new_notes
        total_notes += original_count
        removed_notes += (original_count - len(new_notes))

    print(f"[Smart Filter] Removed {removed_notes} notes out of {total_notes}.")
    return midi_data



In [ ]:
midi3 = smart_filter_short_notes(merge_instruments(pretty_midi.PrettyMIDI(midifile)))

midi1 = merge_instruments(pretty_midi.PrettyMIDI(midifile))

midi2 = pretty_midi.PrettyMIDI('/content/drive/MyDrive/AMT/8/8.mid')
for instrument in midi2.instruments:
    for note in instrument.notes:
      note.pitch = note.pitch + 2

combined = pretty_midi.PrettyMIDI()

for inst in midi1.instruments:
    combined.instruments.append(inst)

for inst in midi2.instruments:
    combined.instruments.append(inst)

for inst in midi3.instruments:
    combined.instruments.append(inst)

combined.write('/content/combined3.mid')

In [ ]:
from collections import defaultdict

def merge_short_notes(notes, min_duration, max_gap):
    """
    Merges short notes of the same pitch into one when they are close to each other.
    Internally, it creates arrays for each pitch and processes them independently.
    """

    # --- STEP 1: Create an array for each pitch (grouping) ---
    notes_by_pitch = defaultdict(list)
    for note in notes:
        # Group notes into a dictionary where the key is the pitch number
        notes_by_pitch[note.pitch].append(note)

    final_merged_notes = []

    # --- STEP 2: Process each pitch independently ---
    for pitch, pitch_notes in notes_by_pitch.items():

        # 1. Sort notes within this pitch by start time
        sorted_notes = sorted(pitch_notes, key=lambda x: x.start)

        if not sorted_notes:
            continue
        if pitch == 40:
          for note in sorted_notes:
            print(note.start, note.end)

        # Take the first note as the starting note for this pitch
        current_note = sorted_notes[0]

        for next_note in sorted_notes[1:]:

            if next_note.end <= current_note.end:
              continue
            else:
              if current_note.end >= next_note.start:
                current_note.end = next_note.end
                continue
            # Compute the duration of the current and next notes
            curr_dur = current_note.end - current_note.start
            next_dur = next_note.end - next_note.start

            # Compute the gap between the end of the current note and the start of the next one
            # (the gap can be negative if the notes overlap)
            gap = next_note.start - current_note.end

            # Conditions for merging:
            # 1. (same_pitch) is automatically satisfied because we are inside one pitch group
            # 2. The gap is small
            # 3. At least one of the notes is short
            close_enough = (gap <= max_gap)
            is_short = (curr_dur < min_duration or next_dur < min_duration)

            if close_enough and is_short:
                # Merge: extend the current note to the end of the next one
                current_note.end = max(current_note.end, next_note.end)

            else:
                # If we do not merge, save the current note and move on to the next one
                final_merged_notes.append(current_note)
                current_note = next_note

        # Add the last note being processed for this pitch
        final_merged_notes.append(current_note)

    # Sort the final list by time to restore the correct note order
    final_merged_notes.sort(key=lambda x: x.start)

    return final_merged_notes

In [ ]:

midi1 = merge_instruments(pretty_midi.PrettyMIDI(midifile))

midi2 = merge_instruments(pretty_midi.PrettyMIDI(midifile))
for instrument in midi2.instruments:
    for note in instrument.notes:
      note.pitch = note.pitch + 2

combined = pretty_midi.PrettyMIDI()

for inst in midi1.instruments:
    combined.instruments.append(inst)

combined.instruments[0].notes = merge_short_notes(midi1.instruments[0].notes, 0.1, 0.05)

for inst in midi2.instruments:
    combined.instruments.append(inst)



combined.write('/content/combined4.mid')

In [ ]:

for instrument in est_midi.instruments:
    print("Instrument program:", instrument.program)
    for note in instrument.notes:
      if(note.start > 290 and note.end < 300):
        note_name = pitch_to_note(note.pitch)
        print(f"Note: name={note_name}, pitch={note.pitch}, start={note.start:.2f}, end={note.end:.2f}, velocity={note.velocity}")

note=F♯3, pitch=54 start=21.79, end=21.94, velocity=37

In [ ]:
import numpy as np
import librosa





# Load the audio
y, sr = librosa.load('/content/drive/MyDrive/AMT/8/8.wav')
# sr = 16000
hop_length=128


# Compute the Constant-Q Transform (CQT) ? it is better suited for music,
# because the frequencies correspond to musical steps (MIDI notes)
# This gives us an "energy image" over time for each pitch
print("Computing the CQT spectrogram...")
cqt = np.abs(librosa.cqt(y, sr=sr, hop_length=hop_length, fmin=librosa.note_to_hz('C1')))
times = librosa.frames_to_time(range(cqt.shape[1]), sr=sr, hop_length=hop_length)


def _find_offset_in_audio(pitch, start_time, end_time_est, plot):

    """
    """Finds the point where the energy falls below the threshold for a specific note."""
    """
    # Note frequency in Hz
    freq = librosa.midi_to_hz(pitch)

    # Index of the CQT bin (approximate)
    # CQT uses a logarithmic scale, so we need the nearest index
    cqt_freqs = librosa.cqt_frequencies(cqt.shape[0], fmin=librosa.note_to_hz('C1'))
    bin_idx = np.argmin(np.abs(cqt_freqs - freq))

    # Frame range for the search
    start_frame = np.searchsorted(times, start_time)
    # Search a bit past the estimated end in case the model cut the note too early
    end_frame_search = np.searchsorted(times, end_time_est + 1.0)

    if start_frame >= end_frame_search:
        return end_time_est

    # Take the energy slice at the frequency of this note
    energy_slice = cqt[bin_idx, start_frame:end_frame_search]

    if energy_slice.size == 0:
        return end_time_est

    # Threshold: a percentage of the maximum energy at the start of the note
    # Use the maximum at the start (attack)
    peak_energy = np.max(energy_slice[:10])
    threshold = peak_energy * thresh_ratio

    # Find the first frame where the energy drops below the threshold
    # Move from the start toward the end
    silent_frames = np.where(energy_slice < threshold)[0]

    # if len(silent_frames) > 0:
    #     # Take the first drop point
    #     first_silent_rel_idx = silent_frames[0]
    #     offset_time = times[start_frame + first_silent_rel_idx]
    #     return offset_time
    # else:
    #     # If the energy never drops, return the model estimate plus a margin
    #     return end_time_est


    if len(silent_frames) > 0:
        first_silent_rel_idx = silent_frames[0]
        real_end_time = times[start_frame + first_silent_rel_idx]
    else:
        # If the energy never drops, keep the estimate as is, or extend it?
        # Here we keep the model estimate (or we could return the end of the search interval)
        pass

    # ---------------------------------------------------------
    # VISUALIZATION BLOCK
    # ---------------------------------------------------------
    if plot:
        import matplotlib.pyplot as plt

        # Create the time axis for the X axis (only for the displayed slice)
        time_axis = times[start_frame:end_frame_search]

        plt.figure(figsize=(12, 5))

        # Plot 1: signal energy
        plt.plot(time_axis, energy_slice, label=f'Energy (Pitch {pitch} / {freq:.1f} Hz)', color='blue', linewidth=1.5)

        # Plot 2: silence threshold
        plt.axhline(y=threshold, color='red', linestyle='--', linewidth=1.5, label=f'Noise Threshold ({thresh_ratio*100:.0f}% of peak)')

        # Marker 3: model prediction (Model End)
        plt.axvline(x=end_time_est, color='green', linestyle=':', linewidth=2, label='Model Offset (Original End)')

        # Marker 4: corrected end (Corrected Offset)
        plt.axvline(x=real_end_time, color='orange', linestyle='-', linewidth=2, label='Corrected Offset (Real End)')

        plt.axvline(x=21.47, color='orange', linestyle='-', linewidth=1)
        plt.axvline(x=21.63, color='orange', linestyle='-', linewidth=1)


        # View settings
        plt.title(f"Offset Correction Analysis | Note Start: {start_time:.2f}s")
        plt.xlabel("Time (s)")
        plt.ylabel("Amplitude (Energy)")
        plt.legend(loc='upper right')
        plt.grid(True, alpha=0.3)

        # Expand the visible range so the details around the note ending are easier to inspect
        plot_start = max(0, start_time - 0.2)
        plot_end = max(real_end_time, end_time_est) + 0.5
        plt.xlim(plot_start, plot_end)

        plt.show()



In [ ]:

thresh_ratio = 0.9
_find_offset_in_audio(59, 290, 291, True)

In [ ]:

thresh_ratio = 0.9
_find_offset_in_audio(42, 21.40, 22.53, True)

In [ ]:

thresh_ratio = 0.9
_find_offset_in_audio(49, 21.60, 22.53, True)

In [ ]:
y, sr = librosa.load('/content/11.wav')

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

def plot_specific_frequency_strength(pitch: int,
                                     audio: np.ndarray,
                                     sr: int,
                                     note_start: float,
                                     note_end: float,
                                     plot: bool = True):
    """
    Demonstrates the strength of this exact frequency (f0) over time.
    Returns: times, strength_db (in dB) ? an array of strengths for further analysis.
    """
    f0 = librosa.midi_to_hz(pitch)          # exact note frequency
    window_length = 1024
    hop_length = 256

    # Extract the note segment + 200 ms buffer (to reveal the decay)
    start_idx = int(note_start * sr)
    end_idx   = int((note_end + 0.2) * sr)
    y_note = audio[start_idx:end_idx]

    # STFT ? compute the spectrogram
    D = librosa.stft(y_note, n_fft=window_length, hop_length=hop_length,
                     window='hann', center=True)
    mag = np.abs(D) ** 2                    # power spectrum

    # Find the nearest bin to our f0
    freqs = librosa.fft_frequencies(sr=sr, n_fft=window_length)
    bin_idx = np.argmin(np.abs(freqs - f0))

    # Strength of this exact frequency across frames (you can average ?1 bin for stability)
    strength = mag[bin_idx, :]
    if bin_idx > 0 and bin_idx < len(freqs)-1:
        strength = mag[bin_idx-1:bin_idx+2, :].mean(axis=0)  # ?1 bin (~10-20 Hz)

    # Time axis and conversion to dB
    times = librosa.times_like(strength, sr=sr, hop_length=hop_length)
    strength_db = librosa.power_to_db(strength, ref=np.max(strength))

    if plot:
        plt.figure(figsize=(10, 5))
        plt.plot(times + note_start, strength_db, label=f'f₀ = {f0:.1f} Hz (pitch {pitch})',
                 color='red', linewidth=2)
        plt.axvline(note_start, color='green', linestyle='--', label='onset (predicted)')
        plt.axvline(note_end, color='blue', linestyle='--', label='rough offset (YourMT3)')
        plt.xlabel('Time, s')
        plt.ylabel('Frequency strength, dB (relative to the maximum)')
        plt.title(f'Frequency strength for {pitch_to_note(pitch)} (pitch={pitch})')
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

    return times + note_start, strength_db

In [ ]:
# 21.4679	22.5709	42
# 21.6259	22.5709	49
# 21.7909	22.5709	54
# 21.9449	22.5709	57
# 22.1089	22.5709	61
# 22.2679	22.5709	66


plot_specific_frequency_strength(42, y, sr, 21.69, 22.53, True )

In [ ]:
import pretty_midi
import numpy as np

def midi_to_freq(pitch):
    return 440.0 * (2 ** ((pitch - 69) / 12.0))

def remove_overtones(notes, time_overlap_threshold=0.3, strength_ratio=1.5, max_harmonic=5):
    """
    notes: list of dicts or namedtuple: {'onset': , 'offset': , 'pitch': , 'velocity': , ...}
    """
    # notes = sorted(notes, key=lambda x: (x['onset'], -x['velocity']))  # strongest first
    kept = []

    for note in notes:
        f = midi_to_freq(note['pitch'])
        is_harmonic = False

        for kept_note in kept:
            if kept_note['offset'] <= note['onset'] or kept_note['onset'] >= note['offset']:
                continue  # no overlap

            f_low = midi_to_freq(kept_note['pitch'])
            ratio = f / f_low
            harmonic_order = round(ratio)

            if 1.95 <= ratio <= harmonic_order + 0.05 and 2 <= harmonic_order <= max_harmonic:
                overlap = min(note['offset'], kept_note['offset']) - max(note['onset'], kept_note['onset'])
                if overlap / (note['offset'] - note['onset']) > time_overlap_threshold:
                    # compare "strength" (velocity ? duration)
                    strength_new = note['velocity'] * (note['offset'] - note['onset'])
                    strength_low = kept_note['velocity'] * (kept_note['offset'] - kept_note['onset'])
                    if strength_new < strength_low / strength_ratio:
                        is_harmonic = True
                        break

        if not is_harmonic:
            kept.append(note)

    return kept

In [ ]:
print(remove_overtones(midi2.instruments[0].notes))

In [ ]:
print(midi_to_freq(87))
print(midi_to_freq(66))
print(midi_to_freq(70))
print(midi_to_freq(75))
print(midi_to_freq(82))


In [ ]:
print(midi_to_freq(58))

In [ ]:
Note: pitch=87, start=12.15, end=14.32, velocity=100
Note: pitch=66, start=12.15, end=14.34, velocity=100
Note: pitch=70, start=12.15, end=14.34, velocity=100
Note: pitch=75, start=12.15, end=14.34, velocity=100
Note: pitch=82, start=12.15, end=14.34, velocity=100

In [ ]:
def midi_pitch_to_note(pitch: int) -> str:
    """
    Converts a MIDI pitch (0-127) into a note label (e.g., 69 -> 'A4').

    Args:
        pitch (int): MIDI note number (0-127).

    Returns:
        str: Note name with octave (e.g., 'C4', 'F#3').

    Raises:
        ValueError: if pitch is outside the valid range.
    """
    if not 0 <= pitch <= 127:
        raise ValueError("MIDI pitch must be in the range 0-127")

    note_names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    note_index = pitch % 12
    octave = (pitch // 12) - 1  # MIDI 0 = C-1, 12 = C0, ..., 60 = C4, 69 = A4
    return f"{note_names[note_index]}{octave}"

In [ ]:
print(midi_pitch_to_note(87))
print(midi_pitch_to_note(66))
print(midi_pitch_to_note(70))
print(midi_pitch_to_note(75))
print(midi_pitch_to_note(82))

In [ ]:
def estimate_velocity(midi, audio, sr):
    for inst in midi.instruments:
        for note in inst.notes:
            y = audio[int(note.start*sr):int(note.end*sr)]
            rms = librosa.feature.rms(y=y)[0].max()
            # normalize across the entire track
            note.velocity = int(np.clip(rms * 127 / global_max_rms, 20, 127))  # floor 20
    return midi



---



In [ ]:
import librosa
import numpy as np

# ────────────────────────────────────────────────
# Load the WAV file once at the start of the script
# ────────────────────────────────────────────────

file_path = "/content/MAPS_MUS-ty_november_ENSTDkAm.wav"

# mono = True  ? always use a single channel (very convenient for AMT)
# sr = None    ? keep the file's native sample rate
y, sr = librosa.load(file_path, sr=None, mono=True)

# Now y is a numpy array that can be passed into any function
print("Signal length:", len(y))
print("Sample rate:", sr)
print("Data type:", y.dtype)          # usually float32
print("Maximum amplitude:", np.max(np.abs(y)))


In [ ]:
import librosa
import numpy as np

def estimate_velocities_from_onsets(y, sr, hop_length=512):
    # 1. Compute onset strength (a very useful librosa function)
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop_length)

    # 2. Find note onsets
    onset_frames = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr)
    onset_times = librosa.frames_to_time(onset_frames, sr=sr, hop_length=hop_length)

    # 3. RMS across the whole file (for normalization)
    rms_all = librosa.feature.rms(y=y, hop_length=hop_length)[0]
    rms_max = np.max(rms_all) + 1e-8  # avoid division by zero

    velocities = []

    for frame in onset_frames:
        # take the RMS value at the onset point plus a few neighboring frames
        start = max(0, frame - 2)
        end   = min(len(rms_all), frame + 6)
        local_rms = np.mean(rms_all[start:end])

        # convert to velocity (you can tune the nonlinearity)
        vel = 127 * (local_rms / rms_max) ** 0.5   # 0.7 often gives a more natural result
        vel = int(np.clip(vel, 1, 127))            # almost never 0

        velocities.append(vel)

    return onset_times, velocities, onset_env

In [ ]:
onset_times, velocities, onset_env = estimate_velocities_from_onsets(y, sr)

## FRONT

This section exposes a public Flask app from Google Colab using `ngrok`.

Before launching the app, store your token in one of these places:

- Colab Secret: `NGROK_AUTHTOKEN`
- Environment variable: `NGROK_AUTHTOKEN`

The main site will be available at `/`, and the preserved sample gallery lives at `/demo`.


In [3]:
!pip -q install pyngrok
!pip install pretty_midi


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 76.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 6.4 MB/s eta 0:00:00
  Created wheel for pretty_midi: filename=pretty_midi-0.2.11-py3-none-any.whl size=5595886 sha256=ab15b5df0b426dee8654789a7712ec73bb5a3d97ce452b765a14157f9258de14
  Stored in directory: /root/.cache/pip/wheels/f4/ad/93/a7042fe12668827574927ade9deec7f29aad2a1001b1501882
Successfully built pretty_midi


In [4]:
import mimetypes
import os
import re
import shutil
import tempfile
import threading
import time
import uuid
from pathlib import Path

import pretty_midi
import torchaudio
from flask import Flask, jsonify, render_template_string, request, send_file
from pyngrok import ngrok

try:
    from google.colab import userdata
except ImportError:
    userdata = None


APP_ROOT = Path("/content/amt_app")
UPLOAD_DIR = APP_ROOT / "uploads"
RESULT_DIR = APP_ROOT / "results"
DEMO_BASE_DIR = Path("/content/drive/MyDrive/AMT")
MAX_AUDIO_DURATION_SECONDS = 5 * 60
ALLOWED_EXTENSIONS = {".wav", ".mp3"}
APP_PORT = 5000

APP_ROOT.mkdir(parents=True, exist_ok=True)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = 128 * 1024 * 1024

JOB_STORE = {}
JOB_LOCK = threading.Lock()


def sanitize_stem(name: str) -> str:
    stem = Path(name).stem
    stem = re.sub(r"[^A-Za-z0-9._-]+", "_", stem).strip("._-")
    return stem or "track"


def allowed_audio_file(filename: str) -> bool:
    return Path(filename).suffix.lower() in ALLOWED_EXTENSIONS


def get_audio_duration_seconds(file_path: Path) -> float:
    info = torchaudio.info(str(file_path))
    return float(info.num_frames) / float(info.sample_rate)


def midi_metadata(midi_path: Path) -> dict:
    midi = pretty_midi.PrettyMIDI(str(midi_path))
    notes = [note for instrument in midi.instruments for note in instrument.notes]
    pitches = [note.pitch for note in notes]
    durations = [note.end - note.start for note in notes]
    programs = sorted(
        {
            instrument.program
            for instrument in midi.instruments
            if not instrument.is_drum
        }
    )
    return {
        "duration_seconds": round(float(midi.get_end_time()), 2),
        "note_count": len(notes),
        "track_count": len(midi.instruments),
        "pitch_min": min(pitches) if pitches else None,
        "pitch_max": max(pitches) if pitches else None,
        "average_note_duration": round(sum(durations) / len(durations), 3) if durations else 0,
        "programs": programs,
    }


def audio_metadata(file_path: Path) -> dict:
    info = torchaudio.info(str(file_path))
    return {
        "sample_rate": int(info.sample_rate),
        "channels": int(info.num_channels),
        "duration_seconds": round(float(info.num_frames) / float(info.sample_rate), 2),
        "encoding": str(info.encoding).lower(),
        "bits_per_sample": int(info.bits_per_sample) if info.bits_per_sample else None,
        "filesize_mb": round(file_path.stat().st_size / (1024 * 1024), 2),
    }


def guess_mimetype(file_path: Path) -> str:
    mime, _ = mimetypes.guess_type(str(file_path))
    return mime or "application/octet-stream"


def ensure_runtime_ready() -> None:
    missing = [
        name for name in ("model", "prepare_media", "transcribe")
        if name not in globals()
    ]
    if missing:
        joined = ", ".join(missing)
        raise RuntimeError(
            "Frontend prerequisites are missing. Run the setup/helper/model cells first. "
            f"Missing objects: {joined}."
        )


def purge_jobs(keep_job_id: str | None = None) -> None:
    stale_job_ids = []
    for job_id, job in list(JOB_STORE.items()):
        if keep_job_id is not None and job_id == keep_job_id:
            continue
        try:
            shutil.rmtree(job["dir"], ignore_errors=True)
        finally:
            stale_job_ids.append(job_id)
    for job_id in stale_job_ids:
        JOB_STORE.pop(job_id, None)


def register_job(audio_path: Path, midi_path: Path, original_name: str) -> dict:
    job_id = uuid.uuid4().hex[:12]
    job_dir = RESULT_DIR / job_id
    job_dir.mkdir(parents=True, exist_ok=True)

    stored_audio = job_dir / f"source{audio_path.suffix.lower()}"
    stored_midi = job_dir / f"{sanitize_stem(original_name)}.mid"
    shutil.copy2(audio_path, stored_audio)
    shutil.copy2(midi_path, stored_midi)

    payload = {
        "id": job_id,
        "dir": job_dir,
        "audio_path": stored_audio,
        "midi_path": stored_midi,
        "created_at": time.time(),
        "display_name": Path(original_name).name,
        "audio_meta": audio_metadata(stored_audio),
        "midi_meta": midi_metadata(stored_midi),
    }

    with JOB_LOCK:
        purge_jobs()
        JOB_STORE[job_id] = payload
    return payload


def sample_files(sample_id: int) -> tuple[Path, Path]:
    sample_dir = DEMO_BASE_DIR / str(sample_id)
    if not sample_dir.exists():
        raise FileNotFoundError(f"Sample folder {sample_id} was not found.")

    audio_files = sorted(
        [p for p in sample_dir.iterdir() if p.suffix.lower() in {".wav", ".mp3", ".ogg"}]
    )
    midi_files = sorted(
        [p for p in sample_dir.iterdir() if p.suffix.lower() in {".mid", ".midi"}]
    )
    if not audio_files or not midi_files:
        raise FileNotFoundError(f"Sample {sample_id} does not contain both audio and MIDI.")
    return audio_files[0], midi_files[0]


def sample_payload(sample_id: int) -> dict:
    audio_path, midi_path = sample_files(sample_id)
    return {
        "sample_id": sample_id,
        "title": f"Sample {sample_id}",
        "audio_url": f"/demo/media/{sample_id}/{audio_path.name}",
        "midi_url": f"/demo/media/{sample_id}/{midi_path.name}",
        "download_url": f"/demo/media/{sample_id}/{midi_path.name}?download=1",
        "audio_meta": audio_metadata(audio_path),
        "midi_meta": midi_metadata(midi_path),
        "files": [
            {"name": audio_path.name, "kind": "audio"},
            {"name": midi_path.name, "kind": "midi"},
        ],
    }


APP_STYLE = """
<style>
  @import url('https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@400;500;700&family=IBM+Plex+Sans:wght@400;500;600&display=swap');

  :root {
    --bg: #08111f;
    --panel: rgba(8, 17, 31, 0.74);
    --panel-strong: rgba(11, 24, 44, 0.92);
    --line: rgba(144, 181, 255, 0.18);
    --text: #edf3ff;
    --muted: #8aa1c7;
    --cyan: #79f2ff;
    --coral: #ff866e;
    --gold: #ffd36b;
    --green: #7df0b8;
    --shadow: 0 28px 80px rgba(0, 0, 0, 0.35);
  }

  * { box-sizing: border-box; }
  html, body { margin: 0; min-height: 100%; }
  body {
    font-family: 'IBM Plex Sans', sans-serif;
    background:
      radial-gradient(circle at 12% 18%, rgba(121, 242, 255, 0.16), transparent 28%),
      radial-gradient(circle at 88% 12%, rgba(255, 134, 110, 0.18), transparent 24%),
      radial-gradient(circle at 50% 84%, rgba(255, 211, 107, 0.11), transparent 22%),
      linear-gradient(180deg, #050b16 0%, #0a1629 52%, #071120 100%);
    color: var(--text);
  }

  body::before {
    content: "";
    position: fixed;
    inset: 0;
    pointer-events: none;
    background-image:
      linear-gradient(rgba(255,255,255,0.025) 1px, transparent 1px),
      linear-gradient(90deg, rgba(255,255,255,0.025) 1px, transparent 1px);
    background-size: 42px 42px;
    mask-image: linear-gradient(180deg, rgba(255,255,255,.24), rgba(255,255,255,.04));
  }

  a { color: inherit; text-decoration: none; }
  .page {
    width: min(1200px, calc(100vw - 32px));
    margin: 0 auto;
    padding: 28px 0 48px;
  }

  .topbar, .hero, .panel, .result-shell, .demo-shell {
    background: var(--panel);
    border: 1px solid var(--line);
    backdrop-filter: blur(18px);
    box-shadow: var(--shadow);
  }

  .topbar {
    border-radius: 24px;
    padding: 18px 22px;
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 22px;
  }

  .brand {
    display: flex;
    flex-direction: column;
    gap: 4px;
  }

  .eyebrow {
    text-transform: uppercase;
    letter-spacing: 0.18em;
    font-size: 0.72rem;
    color: var(--cyan);
  }

  .brand strong, h1, h2, h3 {
    font-family: 'Space Grotesk', sans-serif;
  }

  .nav {
    display: flex;
    gap: 10px;
    flex-wrap: wrap;
  }

  .nav a {
    padding: 10px 16px;
    border: 1px solid rgba(121, 242, 255, 0.14);
    border-radius: 999px;
    color: var(--muted);
    transition: 180ms ease;
  }

  .nav a.active, .nav a:hover {
    color: var(--text);
    background: rgba(121, 242, 255, 0.08);
    border-color: rgba(121, 242, 255, 0.4);
  }

  .hero {
    border-radius: 36px;
    padding: 34px;
    display: grid;
    grid-template-columns: minmax(0, 1.25fr) minmax(280px, 0.75fr);
    gap: 26px;
    margin-bottom: 24px;
  }

  .hero h1 {
    margin: 12px 0 14px;
    font-size: clamp(2.5rem, 4vw, 4.8rem);
    line-height: 0.98;
  }

  .hero p {
    color: var(--muted);
    max-width: 62ch;
    font-size: 1.03rem;
  }

  .hero-copy {
    display: flex;
    flex-direction: column;
    justify-content: space-between;
    gap: 18px;
  }

  .hero-stats {
    display: grid;
    grid-template-columns: repeat(3, minmax(0, 1fr));
    gap: 12px;
  }

  .stat {
    border: 1px solid rgba(255,255,255,0.08);
    border-radius: 20px;
    padding: 16px;
    background: rgba(255,255,255,0.03);
  }

  .stat label {
    display: block;
    font-size: 0.78rem;
    color: var(--muted);
    margin-bottom: 6px;
    text-transform: uppercase;
    letter-spacing: 0.1em;
  }

  .stat strong {
    font-size: 1.2rem;
  }

  .callout {
    border-radius: 28px;
    padding: 24px;
    background:
      linear-gradient(160deg, rgba(121, 242, 255, 0.12), rgba(121, 242, 255, 0.03)),
      rgba(255,255,255,0.02);
    border: 1px solid rgba(121, 242, 255, 0.16);
    display: flex;
    flex-direction: column;
    justify-content: space-between;
    gap: 18px;
  }

  .callout strong {
    font-size: 1rem;
  }

  .callout p {
    margin: 0;
    color: var(--muted);
  }

  .pill-row {
    display: flex;
    flex-wrap: wrap;
    gap: 10px;
  }

  .pill {
    display: inline-flex;
    align-items: center;
    gap: 8px;
    padding: 10px 14px;
    border-radius: 999px;
    background: rgba(255,255,255,0.04);
    border: 1px solid rgba(255,255,255,0.08);
    color: var(--text);
    font-size: 0.95rem;
  }

  .panel {
    border-radius: 32px;
    padding: 28px;
    margin-bottom: 24px;
  }

  .panel-head {
    display: flex;
    justify-content: space-between;
    align-items: center;
    gap: 16px;
    margin-bottom: 18px;
  }

  .panel-head h2 {
    margin: 0;
    font-size: clamp(1.5rem, 2vw, 2rem);
  }

  .subtle {
    color: var(--muted);
  }

  .upload-shell {
    display: grid;
    grid-template-columns: minmax(0, 1.2fr) minmax(280px, 0.8fr);
    gap: 18px;
  }

  .dropzone {
    position: relative;
    border-radius: 28px;
    border: 1.5px dashed rgba(121, 242, 255, 0.36);
    background: linear-gradient(180deg, rgba(121, 242, 255, 0.07), rgba(255,255,255,0.02));
    min-height: 280px;
    display: flex;
    align-items: center;
    justify-content: center;
    text-align: center;
    padding: 28px;
    overflow: hidden;
  }

  .dropzone input[type=file] {
    position: absolute;
    inset: 0;
    opacity: 0;
    cursor: pointer;
  }

  .dropzone.active {
    border-color: rgba(255, 211, 107, 0.85);
    box-shadow: inset 0 0 0 1px rgba(255, 211, 107, 0.36);
  }

  .dropzone h3 {
    margin: 12px 0 10px;
    font-size: 1.7rem;
  }

  .dropzone p {
    margin: 0 auto;
    max-width: 42ch;
    color: var(--muted);
  }

  .form-side {
    display: flex;
    flex-direction: column;
    gap: 14px;
  }

  .mini-card {
    border-radius: 24px;
    border: 1px solid rgba(255,255,255,0.08);
    padding: 18px;
    background: rgba(255,255,255,0.03);
  }

  .mini-card strong {
    display: block;
    margin-bottom: 6px;
  }

  .status-box {
    border-radius: 22px;
    padding: 16px 18px;
    min-height: 92px;
    background: rgba(255,255,255,0.03);
    border: 1px solid rgba(255,255,255,0.08);
    display: flex;
    flex-direction: column;
    gap: 10px;
  }

  .status-line {
    display: flex;
    align-items: center;
    gap: 10px;
  }

  .dot {
    width: 10px;
    height: 10px;
    border-radius: 50%;
    background: var(--cyan);
    box-shadow: 0 0 0 8px rgba(121, 242, 255, 0.12);
  }

  .actions {
    display: flex;
    flex-wrap: wrap;
    gap: 12px;
  }

  button, .button-link {
    appearance: none;
    border: none;
    border-radius: 16px;
    padding: 13px 18px;
    font: inherit;
    font-weight: 600;
    cursor: pointer;
    transition: transform 180ms ease, box-shadow 180ms ease, background 180ms ease;
  }

  button:hover, .button-link:hover {
    transform: translateY(-1px);
  }

  .button-primary {
    color: #05111d;
    background: linear-gradient(135deg, var(--cyan), var(--gold));
    box-shadow: 0 10px 26px rgba(121, 242, 255, 0.2);
  }

  .button-secondary {
    color: var(--text);
    background: rgba(255,255,255,0.05);
    border: 1px solid rgba(255,255,255,0.1);
  }

  .button-secondary[disabled], .button-primary[disabled] {
    cursor: not-allowed;
    opacity: 0.55;
    transform: none;
  }

  .result-shell, .demo-shell {
    border-radius: 36px;
    padding: 28px;
    margin-top: 24px;
  }

  .result-header {
    display: flex;
    justify-content: space-between;
    align-items: flex-start;
    gap: 18px;
    margin-bottom: 16px;
  }

  .transport {
    display: flex;
    flex-wrap: wrap;
    gap: 10px;
    margin-bottom: 14px;
  }

  .segmented {
    display: inline-flex;
    padding: 6px;
    border-radius: 18px;
    background: rgba(255,255,255,0.04);
    border: 1px solid rgba(255,255,255,0.08);
    gap: 6px;
  }

  .segmented button {
    background: transparent;
    color: var(--muted);
    padding: 10px 14px;
    border-radius: 12px;
  }

  .segmented button.active {
    color: var(--text);
    background: rgba(121, 242, 255, 0.12);
  }

  .seek-wrap {
    display: grid;
    grid-template-columns: 72px minmax(0, 1fr) 72px;
    gap: 12px;
    align-items: center;
    margin-bottom: 16px;
  }

  .seek-wrap input[type=range] {
    width: 100%;
    accent-color: var(--gold);
  }

  .viewer-shell {
    overflow: hidden;
    border-radius: 30px;
    border: 1px solid rgba(255,255,255,0.08);
    background: linear-gradient(180deg, rgba(4, 10, 18, 0.9), rgba(10, 20, 34, 0.95));
  }

  .viewer-meta {
    display: flex;
    justify-content: space-between;
    gap: 14px;
    padding: 14px 16px;
    border-bottom: 1px solid rgba(255,255,255,0.08);
    color: var(--muted);
    font-size: 0.94rem;
  }

  .viewer-scroll {
    overflow-x: auto;
    overflow-y: hidden;
    position: relative;
  }

  .viewer-canvas {
    display: block;
  }

  .meta-grid {
    display: grid;
    grid-template-columns: repeat(4, minmax(0, 1fr));
    gap: 12px;
    margin-top: 18px;
  }

  .meta-card {
    border-radius: 22px;
    padding: 16px;
    background: rgba(255,255,255,0.03);
    border: 1px solid rgba(255,255,255,0.08);
  }

  .meta-card label {
    display: block;
    color: var(--muted);
    font-size: 0.78rem;
    letter-spacing: 0.08em;
    text-transform: uppercase;
    margin-bottom: 8px;
  }

  .meta-card strong {
    font-size: 1.1rem;
  }

  .file-list {
    display: flex;
    flex-wrap: wrap;
    gap: 10px;
    margin-top: 16px;
  }

  .note {
    color: var(--muted);
    font-size: 0.94rem;
  }

  .demo-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(130px, 1fr));
    gap: 12px;
    margin: 0 0 20px;
  }

  .demo-tile {
    border-radius: 20px;
    background: rgba(255,255,255,0.03);
    border: 1px solid rgba(255,255,255,0.08);
    color: var(--text);
    min-height: 88px;
  }

  .demo-tile.active {
    background: linear-gradient(135deg, rgba(121, 242, 255, 0.16), rgba(255, 211, 107, 0.16));
    border-color: rgba(121, 242, 255, 0.4);
  }

  .empty-state {
    border-radius: 26px;
    padding: 26px;
    background: rgba(255,255,255,0.03);
    border: 1px solid rgba(255,255,255,0.08);
    text-align: center;
  }

  audio {
    width: 100%;
    margin-top: 16px;
    filter: hue-rotate(5deg) saturate(1.1);
  }

  .footer-note {
    margin-top: 20px;
    color: var(--muted);
    font-size: 0.9rem;
  }

  @media (max-width: 920px) {
    .hero, .upload-shell {
      grid-template-columns: 1fr;
    }
    .hero-stats, .meta-grid {
      grid-template-columns: repeat(2, minmax(0, 1fr));
    }
  }

  @media (max-width: 640px) {
    .page {
      width: min(100vw - 20px, 1000px);
      padding-top: 18px;
    }
    .topbar {
      align-items: flex-start;
      flex-direction: column;
    }
    .hero, .panel, .result-shell, .demo-shell {
      padding: 20px;
      border-radius: 24px;
    }
    .hero-stats, .meta-grid {
      grid-template-columns: 1fr;
    }
    .seek-wrap {
      grid-template-columns: 60px minmax(0, 1fr) 60px;
    }
  }
</style>
"""


APP_SCRIPT = """
<script src="https://cdn.jsdelivr.net/npm/tone@15.0.4/build/Tone.js"></script>
<script src="https://cdn.jsdelivr.net/npm/@tonejs/midi@2.0.28/build/Midi.js"></script>
<script>
  function formatSeconds(seconds) {
    const total = Math.max(0, Math.floor(seconds || 0));
    const mins = Math.floor(total / 60).toString();
    const secs = (total % 60).toString().padStart(2, '0');
    return `${mins}:${secs}`;
  }

  function midiNumberToName(number) {
    const names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B'];
    const octave = Math.floor(number / 12) - 1;
    return `${names[number % 12]}${octave}`;
  }

  class PianoRollPlayer {
    constructor(root) {
      this.root = root;
      this.canvas = root.querySelector('[data-roll-canvas]');
      this.ctx = this.canvas.getContext('2d');
      this.scroll = root.querySelector('[data-roll-scroll]');
      this.seek = root.querySelector('[data-seek]');
      this.currentTimeEl = root.querySelector('[data-current-time]');
      this.totalTimeEl = root.querySelector('[data-total-time]');
      this.transportLabel = root.querySelector('[data-transport-label]');
      this.modeButtons = [...root.querySelectorAll('[data-mode]')];
      this.playButton = root.querySelector('[data-play]');
      this.stopButton = root.querySelector('[data-stop]');
      this.downloadLink = root.querySelector('[data-download]');
      this.audioElement = root.querySelector('[data-audio]');
      this.metaName = root.querySelector('[data-meta-name]');
      this.metaDuration = root.querySelector('[data-meta-duration]');
      this.metaNotes = root.querySelector('[data-meta-notes]');
      this.metaRange = root.querySelector('[data-meta-range]');
      this.metaTracks = root.querySelector('[data-meta-tracks]');
      this.noteList = root.querySelector('[data-file-list]');
      this.emptyState = root.querySelector('[data-empty-state]');
      this.resultBlock = root.querySelector('[data-result-block]');
      this.viewerMeta = root.querySelector('[data-viewer-meta]');
      this.status = root.querySelector('[data-player-status]');

      this.playMode = 'midi';
      this.audioUrl = null;
      this.midiUrl = null;
      this.midi = null;
      this.notes = [];
      this.synth = null;
      this.isPlaying = false;
      this.rafId = null;
      this.pxPerSecond = 150;
      this.noteHeight = 16;
      this.minPitch = 21;
      this.maxPitch = 108;

      this.modeButtons.forEach((button) => {
        button.addEventListener('click', () => this.setPlayMode(button.dataset.mode));
      });
      this.playButton.addEventListener('click', () => this.togglePlayback());
      this.stopButton.addEventListener('click', () => this.stopPlayback());
      this.seek.addEventListener('input', () => this.seekTo(Number(this.seek.value)));
      this.audioElement.addEventListener('pause', () => {
        if (this.playMode !== 'midi' && !this.audioElement.ended) {
          this.isPlaying = false;
        }
      });
    }

    async load(payload) {
      this.stopPlayback();
      this.audioUrl = payload.audio_url;
      this.midiUrl = payload.midi_url;
      this.audioElement.src = this.audioUrl || '';
      this.downloadLink.href = payload.download_url || payload.midi_url || '#';
      this.downloadLink.download = payload.download_name || 'transcription.mid';
      this.metaName.textContent = payload.title || payload.display_name || 'Untitled';
      this.metaDuration.textContent = formatSeconds(payload.midi_meta?.duration_seconds || payload.audio_meta?.duration_seconds || 0);
      this.metaNotes.textContent = String(payload.midi_meta?.note_count ?? 0);
      const low = payload.midi_meta?.pitch_min;
      const high = payload.midi_meta?.pitch_max;
      this.metaRange.textContent = low === null || high === null ? 'N/A' : `${midiNumberToName(low)} - ${midiNumberToName(high)}`;
      this.metaTracks.textContent = String(payload.midi_meta?.track_count ?? 0);
      this.noteList.innerHTML = '';
      (payload.files || []).forEach((item) => {
        const tag = document.createElement('span');
        tag.className = 'pill';
        tag.textContent = `${item.kind.toUpperCase()} - ${item.name}`;
        this.noteList.appendChild(tag);
      });
      this.viewerMeta.textContent = 'Rendering MIDI roll...';
      this.status.textContent = 'Result ready. Choose playback mode or scroll through the roll.';
      this.emptyState.hidden = true;
      this.resultBlock.hidden = false;

      const midiResponse = await fetch(this.midiUrl);
      const midiBuffer = await midiResponse.arrayBuffer();
      this.midi = new Midi(midiBuffer);
      this.notes = this.midi.tracks.flatMap((track, trackIndex) =>
        track.notes.map((note) => ({ ...note, trackIndex }))
      );

      if (this.notes.length) {
        this.minPitch = Math.min(...this.notes.map((note) => note.midi));
        this.maxPitch = Math.max(...this.notes.map((note) => note.midi));
      } else {
        this.minPitch = 48;
        this.maxPitch = 72;
      }

      const duration = this.durationSeconds();
      this.seek.max = String(duration || 1);
      this.seek.value = '0';
      this.totalTimeEl.textContent = formatSeconds(duration);
      this.currentTimeEl.textContent = formatSeconds(0);
      this.renderRoll(0);
    }

    durationSeconds() {
      if (!this.midi) return 0;
      return Math.max(this.midi.duration || 0, this.audioElement.duration || 0, 0.1);
    }

    ensureSynth() {
      if (!this.synth) {
        this.synth = new Tone.PolySynth(Tone.Synth, {
          oscillator: { type: 'triangle8' },
          envelope: { attack: 0.01, decay: 0.08, sustain: 0.2, release: 0.45 }
        }).toDestination();
        this.synth.volume.value = -8;
      }
    }

    setPlayMode(mode) {
      this.playMode = mode;
      this.modeButtons.forEach((button) => button.classList.toggle('active', button.dataset.mode === mode));
      const labels = {
        audio: 'Playback mode: source audio only',
        midi: 'Playback mode: generated MIDI only',
        sync: 'Playback mode: audio + MIDI together'
      };
      this.transportLabel.textContent = labels[mode];
    }

    async scheduleMidi(startAt = 0) {
      if (!this.midi) return;
      await Tone.start();
      this.ensureSynth();
      Tone.Transport.stop();
      Tone.Transport.cancel();
      Tone.Transport.seconds = startAt;
      this.midi.tracks.forEach((track) => {
        track.notes.forEach((note) => {
          Tone.Transport.schedule((time) => {
            this.synth.triggerAttackRelease(note.name, note.duration, time, note.velocity);
          }, note.time);
        });
      });
    }

    async togglePlayback() {
      if (!this.midi) return;
      if (this.isPlaying) {
        this.pausePlayback();
        return;
      }

      const startAt = Number(this.seek.value || 0);
      await this.scheduleMidi(startAt);
      if (this.playMode === 'audio' || this.playMode === 'sync') {
        this.audioElement.currentTime = startAt;
      }

      if (this.playMode === 'audio') {
        await this.audioElement.play();
      } else if (this.playMode === 'midi') {
        Tone.Transport.start(undefined, startAt);
      } else {
        await this.audioElement.play();
        Tone.Transport.start(undefined, startAt);
      }

      this.isPlaying = true;
      this.playButton.textContent = 'Pause';
      this.tick();
    }

    pausePlayback() {
      if (this.playMode === 'audio' || this.playMode === 'sync') {
        this.audioElement.pause();
      }
      if (this.playMode === 'midi' || this.playMode === 'sync') {
        Tone.Transport.pause();
      }
      this.isPlaying = false;
      this.playButton.textContent = 'Play';
      cancelAnimationFrame(this.rafId);
    }

    stopPlayback() {
      this.audioElement.pause();
      this.audioElement.currentTime = 0;
      Tone.Transport.stop();
      Tone.Transport.cancel();
      this.isPlaying = false;
      this.playButton.textContent = 'Play';
      this.seek.value = '0';
      this.currentTimeEl.textContent = formatSeconds(0);
      cancelAnimationFrame(this.rafId);
      if (this.midi) {
        this.renderRoll(0);
      }
    }

    seekTo(seconds) {
      if (this.playMode !== 'midi') {
        this.audioElement.currentTime = seconds;
      }
      if (this.playMode === 'midi' && this.isPlaying) {
        this.pausePlayback();
      }
      this.currentTimeEl.textContent = formatSeconds(seconds);
      this.renderRoll(seconds);
    }

    currentTime() {
      if (!this.isPlaying) return Number(this.seek.value || 0);
      if (this.playMode === 'audio') return this.audioElement.currentTime || 0;
      if (this.playMode === 'midi') return Tone.Transport.seconds || 0;
      return Math.max(this.audioElement.currentTime || 0, Tone.Transport.seconds || 0);
    }

    tick() {
      const current = this.currentTime();
      const duration = this.durationSeconds();
      this.seek.value = String(Math.min(current, duration));
      this.currentTimeEl.textContent = formatSeconds(current);
      this.renderRoll(current);

      const audioEnded = this.playMode !== 'midi' && this.audioElement.ended;
      const midiEnded = this.playMode !== 'audio' && current >= duration;
      if ((this.playMode === 'audio' && audioEnded) ||
          (this.playMode === 'midi' && midiEnded) ||
          (this.playMode === 'sync' && audioEnded)) {
        this.stopPlayback();
        return;
      }

      this.rafId = requestAnimationFrame(() => this.tick());
    }

    renderRoll(playheadSeconds) {
      const duration = this.durationSeconds();
      const pitchSpan = Math.max(12, this.maxPitch - this.minPitch + 1);
      const width = Math.max(960, Math.ceil(duration * this.pxPerSecond) + 120);
      const height = pitchSpan * this.noteHeight + 40;
      this.canvas.width = width;
      this.canvas.height = height;

      const ctx = this.ctx;
      ctx.clearRect(0, 0, width, height);
      ctx.fillStyle = '#071322';
      ctx.fillRect(0, 0, width, height);

      for (let row = 0; row < pitchSpan; row++) {
        const pitch = this.maxPitch - row;
        const y = row * this.noteHeight + 20;
        const isBlack = [1, 3, 6, 8, 10].includes(pitch % 12);
        ctx.fillStyle = isBlack ? 'rgba(121, 242, 255, 0.05)' : 'rgba(255,255,255,0.025)';
        ctx.fillRect(0, y, width, this.noteHeight);
      }

      for (let second = 0; second <= Math.ceil(duration); second++) {
        const x = second * this.pxPerSecond + 80;
        ctx.strokeStyle = second % 4 === 0 ? 'rgba(255,255,255,0.16)' : 'rgba(255,255,255,0.06)';
        ctx.beginPath();
        ctx.moveTo(x, 20);
        ctx.lineTo(x, height);
        ctx.stroke();
        ctx.fillStyle = 'rgba(255,255,255,0.5)';
        ctx.font = '12px IBM Plex Sans';
        ctx.fillText(formatSeconds(second), x + 4, 14);
      }

      this.notes.forEach((note) => {
        const x = note.time * this.pxPerSecond + 80;
        const y = (this.maxPitch - note.midi) * this.noteHeight + 22;
        const w = Math.max(3, note.duration * this.pxPerSecond);
        const hue = (note.trackIndex * 48 + note.midi * 3) % 360;
        ctx.fillStyle = `hsla(${hue}, 84%, 68%, 0.82)`;
        ctx.strokeStyle = `hsla(${hue}, 95%, 80%, 0.95)`;
        ctx.lineWidth = 1;
        ctx.beginPath();
        ctx.roundRect(x, y, w, this.noteHeight - 4, 5);
        ctx.fill();
        ctx.stroke();
      });

      const playheadX = playheadSeconds * this.pxPerSecond + 80;
      ctx.strokeStyle = '#ffd36b';
      ctx.lineWidth = 2;
      ctx.beginPath();
      ctx.moveTo(playheadX, 20);
      ctx.lineTo(playheadX, height);
      ctx.stroke();

      if (playheadX > this.scroll.scrollLeft + this.scroll.clientWidth - 120) {
        this.scroll.scrollLeft = playheadX - this.scroll.clientWidth / 2;
      }

      const topLabel = this.notes.length ? `${this.notes.length} notes across ${this.midi.tracks.length} track(s)` : 'No notes detected in MIDI output';
      this.viewerMeta.textContent = topLabel;
    }
  }

  function createStatusCycler(element) {
    let timer = null;
    const steps = [
      'Uploading audio to the Colab runtime...',
      'Validating length and preparing the waveform...',
      'Running transcription through the model...',
      'Building the MIDI roll and packaging the result...'
    ];
    return {
      start() {
        let index = 0;
        element.textContent = steps[index];
        timer = window.setInterval(() => {
          index = (index + 1) % steps.length;
          element.textContent = steps[index];
        }, 2200);
      },
      stop(finalText) {
        if (timer) {
          window.clearInterval(timer);
          timer = null;
        }
        if (finalText) {
          element.textContent = finalText;
        }
      }
    };
  }
</script>
"""


def render_main_page() -> str:
    return """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>YourMT3+ Public Transcription Lab</title>
  __APP_STYLE__
</head>
<body>
  <div class="page">
    <div class="topbar">
      <div class="brand">
        <span class="eyebrow">Public Colab Interface</span>
        <strong>YourMT3+ Transcription Lab</strong>
      </div>
      <nav class="nav">
        <a class="active" href="/">Transcribe</a>
        <a href="/demo">Demo Gallery</a>
      </nav>
    </div>

    <section class="hero">
      <div class="hero-copy">
        <div>
          <span class="eyebrow">Upload a single file and inspect the generated piano roll</span>
          <h1>Turn raw audio into an explorable MIDI performance.</h1>
          <p>
            This public site runs on Google Colab compute and converts one WAV or MP3 file at a time.
            The result page gives you a horizontal piano roll, separate audio and MIDI playback modes,
            downloadable output, and quick metadata about the generated transcription.
          </p>
        </div>
        <div class="pill-row">
          <span class="pill">Input: WAV or MP3</span>
          <span class="pill">Max duration: 5 minutes</span>
          <span class="pill">Open public link via ngrok</span>
        </div>
      </div>
      <aside class="callout">
        <div>
          <strong>Built for public sharing from Colab</strong>
          <p>Use the main page for personal uploads and keep the original curated examples available at <code>/demo</code>.</p>
        </div>
        <div class="hero-stats">
          <div class="stat">
            <label>Playback</label>
            <strong>Audio / MIDI / Sync</strong>
          </div>
          <div class="stat">
            <label>Output</label>
            <strong>Custom browser roll</strong>
          </div>
          <div class="stat">
            <label>Storage</label>
            <strong>Temporary runtime only</strong>
          </div>
        </div>
      </aside>
    </section>

    <section class="panel">
      <div class="panel-head">
        <div>
          <h2>Upload For Transcription</h2>
          <div class="subtle">One file at a time. The previous runtime result is cleared when a new job starts.</div>
        </div>
      </div>

      <div class="upload-shell">
        <label class="dropzone" id="dropzone">
          <input id="audioInput" type="file" accept=".wav,.mp3,audio/wav,audio/mpeg" />
          <div>
            <span class="eyebrow">Drop audio here</span>
            <h3>Choose a WAV or MP3 file</h3>
            <p>The app validates the file, runs transcription in Colab, then draws a horizontal MIDI roll directly in your browser.</p>
          </div>
        </label>

        <div class="form-side">
          <div class="mini-card">
            <strong>Accepted input</strong>
            <div class="subtle">Single <code>.wav</code> or <code>.mp3</code> file, up to 5 minutes long.</div>
          </div>
          <div class="mini-card">
            <strong>Result package</strong>
            <div class="subtle">Source audio playback, generated MIDI playback, synced mode, metadata, and MIDI download.</div>
          </div>
          <div class="status-box">
            <div class="status-line">
              <span class="dot"></span>
              <strong>Status</strong>
            </div>
            <div class="subtle" id="uploadStatus">Waiting for an audio file.</div>
          </div>
          <div class="actions">
            <button class="button-primary" id="transcribeButton" disabled>Transcribe</button>
            <button class="button-secondary" id="resetButton" type="button">Reset</button>
          </div>
        </div>
      </div>
    </section>

    <section class="result-shell" id="resultShell" data-player-root>
      <div class="empty-state" data-empty-state>
        <h3 style="margin-top:0;">No transcription yet</h3>
        <p class="subtle">Upload a file to generate the first piano roll. The viewer below becomes interactive as soon as the backend returns a MIDI result.</p>
      </div>

      <div hidden data-result-block>
        <div class="result-header">
          <div>
            <span class="eyebrow">Generated result</span>
            <h2 style="margin:10px 0 4px;" data-meta-name>Untitled</h2>
            <div class="subtle" data-player-status>Ready.</div>
          </div>
          <a class="button-link button-secondary" data-download href="#" download="transcription.mid">Download MIDI</a>
        </div>

        <div class="transport">
          <div class="segmented">
            <button class="active" data-mode="midi" type="button">MIDI</button>
            <button data-mode="audio" type="button">Audio</button>
            <button data-mode="sync" type="button">Sync</button>
          </div>
          <button class="button-primary" data-play type="button">Play</button>
          <button class="button-secondary" data-stop type="button">Stop</button>
          <span class="pill" data-transport-label>Playback mode: generated MIDI only</span>
        </div>

        <div class="seek-wrap">
          <span data-current-time>0:00</span>
          <input type="range" min="0" max="1" step="0.01" value="0" data-seek />
          <span data-total-time>0:00</span>
        </div>

        <div class="viewer-shell">
          <div class="viewer-meta" data-viewer-meta>Waiting for MIDI data...</div>
          <div class="viewer-scroll" data-roll-scroll>
            <canvas class="viewer-canvas" data-roll-canvas></canvas>
          </div>
        </div>

        <audio controls preload="metadata" data-audio></audio>

        <div class="meta-grid">
          <div class="meta-card">
            <label>Duration</label>
            <strong data-meta-duration>0:00</strong>
          </div>
          <div class="meta-card">
            <label>Note Count</label>
            <strong data-meta-notes>0</strong>
          </div>
          <div class="meta-card">
            <label>Pitch Range</label>
            <strong data-meta-range>N/A</strong>
          </div>
          <div class="meta-card">
            <label>Tracks</label>
            <strong data-meta-tracks>0</strong>
          </div>
        </div>

        <div class="file-list" data-file-list></div>
        <div class="footer-note">Playback in Sync mode starts the source audio and browser MIDI engine together from the same position.</div>
      </div>
    </section>
  </div>

  __APP_SCRIPT__
  <script>
    const input = document.getElementById('audioInput');
    const transcribeButton = document.getElementById('transcribeButton');
    const resetButton = document.getElementById('resetButton');
    const dropzone = document.getElementById('dropzone');
    const uploadStatus = document.getElementById('uploadStatus');
    const player = new PianoRollPlayer(document.querySelector('[data-player-root]'));
    const cycler = createStatusCycler(uploadStatus);
    let selectedFile = null;

    function setSelectedFile(file) {
      selectedFile = file || null;
      transcribeButton.disabled = !selectedFile;
      if (!selectedFile) {
        uploadStatus.textContent = 'Waiting for an audio file.';
        return;
      }
      uploadStatus.textContent = `Selected: ${selectedFile.name}`;
    }

    function resetUpload() {
      player.stopPlayback();
      input.value = '';
      setSelectedFile(null);
    }

    dropzone.addEventListener('dragenter', () => dropzone.classList.add('active'));
    dropzone.addEventListener('dragleave', () => dropzone.classList.remove('active'));
    dropzone.addEventListener('dragover', (event) => {
      event.preventDefault();
      dropzone.classList.add('active');
    });
    dropzone.addEventListener('drop', (event) => {
      event.preventDefault();
      dropzone.classList.remove('active');
      const file = event.dataTransfer.files?.[0];
      if (file) {
        input.files = event.dataTransfer.files;
        setSelectedFile(file);
      }
    });

    input.addEventListener('change', () => {
      setSelectedFile(input.files?.[0] || null);
    });

    resetButton.addEventListener('click', resetUpload);

    transcribeButton.addEventListener('click', async () => {
      if (!selectedFile) return;

      const formData = new FormData();
      formData.append('audio_file', selectedFile);

      transcribeButton.disabled = true;
      cycler.start();

      try {
        const response = await fetch('/api/transcribe', {
          method: 'POST',
          body: formData,
        });
        const payload = await response.json();
        if (!response.ok) {
          throw new Error(payload.error || 'Transcription failed.');
        }
        cycler.stop('Transcription complete. Loading the player...');
        await player.load(payload);
        uploadStatus.textContent = `Ready: ${payload.display_name}`;
      } catch (error) {
        cycler.stop(`Error: ${error.message}`);
      } finally {
        transcribeButton.disabled = !selectedFile;
      }
    });
  </script>
</body>
</html>
""".replace("__APP_STYLE__", APP_STYLE).replace("__APP_SCRIPT__", APP_SCRIPT)


def render_demo_page() -> str:
    return """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>YourMT3+ Demo Gallery</title>
  __APP_STYLE__
</head>
<body>
  <div class="page">
    <div class="topbar">
      <div class="brand">
        <span class="eyebrow">Public Colab Interface</span>
        <strong>YourMT3+ Demo Gallery</strong>
      </div>
      <nav class="nav">
        <a href="/">Transcribe</a>
        <a class="active" href="/demo">Demo Gallery</a>
      </nav>
    </div>

    <section class="hero">
      <div class="hero-copy">
        <div>
          <span class="eyebrow">Preserved sample endpoint</span>
          <h1>Browse the original examples with the same player.</h1>
          <p>
            The curated sample folders are still available, now rendered with the same interface used for
            uploaded files. Pick any sample to inspect the audio, generated MIDI, metadata, and browser roll.
          </p>
        </div>
        <div class="pill-row">
          <span class="pill">Shared player UI</span>
          <span class="pill">11 retained examples</span>
          <span class="pill">Separate /demo endpoint</span>
        </div>
      </div>
      <aside class="callout">
        <div>
          <strong>Demo route remains independent</strong>
          <p>If the Google Drive sample folder is missing, the page stays up and reports the backend error without breaking the main transcription route.</p>
        </div>
        <div class="hero-stats">
          <div class="stat">
            <label>Route</label>
            <strong>/demo</strong>
          </div>
          <div class="stat">
            <label>Media</label>
            <strong>Audio + MIDI</strong>
          </div>
          <div class="stat">
            <label>Viewer</label>
            <strong>Custom piano roll</strong>
          </div>
        </div>
      </aside>
    </section>

    <section class="demo-shell">
      <div class="panel-head">
        <div>
          <h2>Sample Picker</h2>
          <div class="subtle">Choose a saved example from the mounted Google Drive sample directory.</div>
        </div>
      </div>

      <div class="demo-grid" id="demoGrid"></div>
      <div class="status-box" style="margin-bottom: 18px;">
        <div class="status-line">
          <span class="dot"></span>
          <strong>Status</strong>
        </div>
        <div class="subtle" id="demoStatus">Loading sample 1...</div>
      </div>

      <section class="result-shell" style="margin-top:0;" data-player-root>
        <div class="empty-state" data-empty-state>
          <h3 style="margin-top:0;">No sample loaded</h3>
          <p class="subtle">Pick a sample to populate the shared viewer.</p>
        </div>

        <div hidden data-result-block>
          <div class="result-header">
            <div>
              <span class="eyebrow">Demo result</span>
              <h2 style="margin:10px 0 4px;" data-meta-name>Sample</h2>
              <div class="subtle" data-player-status>Ready.</div>
            </div>
            <a class="button-link button-secondary" data-download href="#" download="sample.mid">Download MIDI</a>
          </div>

          <div class="transport">
            <div class="segmented">
              <button class="active" data-mode="midi" type="button">MIDI</button>
              <button data-mode="audio" type="button">Audio</button>
              <button data-mode="sync" type="button">Sync</button>
            </div>
            <button class="button-primary" data-play type="button">Play</button>
            <button class="button-secondary" data-stop type="button">Stop</button>
            <span class="pill" data-transport-label>Playback mode: generated MIDI only</span>
          </div>

          <div class="seek-wrap">
            <span data-current-time>0:00</span>
            <input type="range" min="0" max="1" step="0.01" value="0" data-seek />
            <span data-total-time>0:00</span>
          </div>

          <div class="viewer-shell">
            <div class="viewer-meta" data-viewer-meta>Waiting for MIDI data...</div>
            <div class="viewer-scroll" data-roll-scroll>
              <canvas class="viewer-canvas" data-roll-canvas></canvas>
            </div>
          </div>

          <audio controls preload="metadata" data-audio></audio>

          <div class="meta-grid">
            <div class="meta-card">
              <label>Duration</label>
              <strong data-meta-duration>0:00</strong>
            </div>
            <div class="meta-card">
              <label>Note Count</label>
              <strong data-meta-notes>0</strong>
            </div>
            <div class="meta-card">
              <label>Pitch Range</label>
              <strong data-meta-range>N/A</strong>
            </div>
            <div class="meta-card">
              <label>Tracks</label>
              <strong data-meta-tracks>0</strong>
            </div>
          </div>

          <div class="file-list" data-file-list></div>
        </div>
      </section>
    </section>
  </div>

  __APP_SCRIPT__
  <script>
    const demoGrid = document.getElementById('demoGrid');
    const demoStatus = document.getElementById('demoStatus');
    const player = new PianoRollPlayer(document.querySelector('[data-player-root]'));

    function renderButtons() {
      for (let i = 1; i <= 11; i += 1) {
        const button = document.createElement('button');
        button.className = 'demo-tile';
        button.type = 'button';
        button.dataset.sampleId = String(i);
        button.innerHTML = `<strong>Sample ${i}</strong><div class="subtle">Open saved pair</div>`;
        button.addEventListener('click', () => loadSample(i));
        demoGrid.appendChild(button);
      }
    }

    async function loadSample(id) {
      demoStatus.textContent = `Loading sample ${id}...`;
      document.querySelectorAll('.demo-tile').forEach((button) => {
        button.classList.toggle('active', button.dataset.sampleId === String(id));
      });

      try {
        const response = await fetch(`/api/demo/${id}`);
        const payload = await response.json();
        if (!response.ok) {
          throw new Error(payload.error || 'Unable to load sample.');
        }
        await player.load(payload);
        demoStatus.textContent = `Loaded sample ${id}.`;
      } catch (error) {
        demoStatus.textContent = `Error: ${error.message}`;
      }
    }

    renderButtons();
    loadSample(1);
  </script>
</body>
</html>
""".replace("__APP_STYLE__", APP_STYLE).replace("__APP_SCRIPT__", APP_SCRIPT)


@app.get("/")
def index():
    return render_template_string(render_main_page())


@app.get("/demo")
def demo():
    return render_template_string(render_demo_page())


@app.post("/api/transcribe")
def api_transcribe():
    ensure_runtime_ready()
    uploaded_file = request.files.get("audio_file")
    if uploaded_file is None or uploaded_file.filename == "":
        return jsonify({"error": "Please upload a WAV or MP3 file."}), 400
    if not allowed_audio_file(uploaded_file.filename):
        return jsonify({"error": "Unsupported file type. Only WAV and MP3 are accepted."}), 400

    temp_dir = Path(tempfile.mkdtemp(dir=UPLOAD_DIR))
    upload_path = temp_dir / uploaded_file.filename

    try:
        uploaded_file.save(upload_path)

        duration_seconds = get_audio_duration_seconds(upload_path)
        if duration_seconds > MAX_AUDIO_DURATION_SECONDS:
            return jsonify({
                "error": f"Audio is too long ({duration_seconds:.1f}s). The limit is 300 seconds."
            }), 400

        prepared = prepare_media(str(upload_path), source_type="audio_filepath")
        midi_path = Path(transcribe(model, prepared))
        if not midi_path.exists():
            raise FileNotFoundError("The transcription step did not produce a MIDI file.")

        job = register_job(upload_path, midi_path, uploaded_file.filename)
        response = {
            "job_id": job["id"],
            "title": sanitize_stem(uploaded_file.filename),
            "display_name": job["display_name"],
            "audio_url": f"/jobs/{job['id']}/audio",
            "midi_url": f"/jobs/{job['id']}/midi",
            "download_url": f"/jobs/{job['id']}/download",
            "download_name": job["midi_path"].name,
            "audio_meta": job["audio_meta"],
            "midi_meta": job["midi_meta"],
            "files": [
                {"name": job["audio_path"].name, "kind": "audio"},
                {"name": job["midi_path"].name, "kind": "midi"},
            ],
        }
        return jsonify(response)
    except Exception as exc:
        return jsonify({"error": str(exc)}), 500
    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)
        if "midi_path" in locals():
            try:
                Path(midi_path).unlink(missing_ok=True)
            except Exception:
                pass


@app.get("/api/demo/<int:sample_id>")
def api_demo(sample_id: int):
    try:
        return jsonify(sample_payload(sample_id))
    except Exception as exc:
        return jsonify({"error": str(exc)}), 404


@app.get("/jobs/<job_id>/audio")
def serve_job_audio(job_id: str):
    job = JOB_STORE.get(job_id)
    if not job:
        return jsonify({"error": "Result not found."}), 404
    return send_file(job["audio_path"], mimetype=guess_mimetype(job["audio_path"]))


@app.get("/jobs/<job_id>/midi")
def serve_job_midi(job_id: str):
    job = JOB_STORE.get(job_id)
    if not job:
        return jsonify({"error": "Result not found."}), 404
    return send_file(job["midi_path"], mimetype="audio/midi")


@app.get("/jobs/<job_id>/download")
def download_job_midi(job_id: str):
    job = JOB_STORE.get(job_id)
    if not job:
        return jsonify({"error": "Result not found."}), 404
    return send_file(job["midi_path"], mimetype="audio/midi", as_attachment=True, download_name=job["midi_path"].name)


@app.get("/demo/media/<int:sample_id>/<path:filename>")
def serve_demo_media(sample_id: int, filename: str):
    sample_dir = DEMO_BASE_DIR / str(sample_id)
    file_path = (sample_dir / filename).resolve()
    if sample_dir.resolve() not in file_path.parents or not file_path.exists():
        return jsonify({"error": "Demo file not found."}), 404
    if request.args.get("download") == "1":
        return send_file(file_path, mimetype=guess_mimetype(file_path), as_attachment=True, download_name=file_path.name)
    return send_file(file_path, mimetype=guess_mimetype(file_path))


@app.get("/health")
def health():
    return jsonify({"status": "ok", "jobs": len(JOB_STORE)})


def resolve_ngrok_token() -> str:
    env_token = os.getenv("NGROK_AUTHTOKEN")
    if env_token:
        return env_token
    if userdata is not None:
        try:
            secret_token = userdata.get("NGROK_AUTHTOKEN")
            if secret_token:
                return secret_token
        except Exception:
            pass
    raise RuntimeError(
        "Set NGROK_AUTHTOKEN in Colab Secrets or as an environment variable before launching the public app."
    )


def launch_public_app(port: int = APP_PORT):
    ensure_runtime_ready()
    token = resolve_ngrok_token()
    ngrok.kill()
    ngrok.set_auth_token(token)
    tunnel = ngrok.connect(port)
    print(f"Public URL: {tunnel.public_url}")
    print(f"Main page: {tunnel.public_url}/")
    print(f"Demo page: {tunnel.public_url}/demo")
    app.run(host="0.0.0.0", port=port, debug=False, use_reloader=False)


In [5]:
launch_public_app()

RuntimeError: Frontend prerequisites are missing. Run the setup/helper/model cells first. Missing objects: model, prepare_media, transcribe.